# Combine and Reshape Datasets

In [1]:
import json
import polars as pl

In [25]:
# Import data from JSON files
df_videos = pl.read_parquet("data/parquet/videos_full.parquet")
df_commentthreads = pl.read_parquet("data/parquet/commentThreads_full.parquet")
df_commentthreads_replies = pl.read_parquet("data/parquet/commentThreadsReplies_full.parquet")

# Clean topics
def clean_topic(topic):
    return topic.lower().replace('-', ' ').replace("'s", '').replace("'", '').replace('.', '').replace('/', ' ').replace('"', '').strip()

# Clean and process the data
df_videos = df_videos.with_columns(
    pl.col('topic').map_elements(clean_topic)
)
df_commentthreads = df_commentthreads.with_columns(
    pl.col('topic').map_elements(clean_topic)
)
df_commentthreads_replies = df_commentthreads_replies.with_columns(
    pl.col('topic').map_elements(clean_topic)
)

/var/folders/pr/x4pd5c4d2y94knncdskg53qm0000gn/T/ipykernel_73259/2270323903.py:12: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("topic").map_elements(clean_topic)
with this one instead:
  + pl.col("topic").str.to_lowercase().str.replace_all('-',' ',literal=True).str.replace_all("'s",'',literal=True).str.replace_all("'",'',literal=True).str.replace_all('.','',literal=True).str.replace_all('/',' ',literal=True).str.replace_all('"','',literal=True).str.strip_chars()

  pl.col('topic').map_elements(clean_topic)
/var/folders/pr/x4pd5c4d2y94knncdskg53qm0000gn/T/ipykernel_73259/2270323903.py:11: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  df_videos = df_videos.with_columns(
/var/folders/pr/x4pd5c4d2y94knnc

In [27]:
# Import BERTopic data
with open("data/BERTopic/us-tariffs_topics_final.json", "r") as f:
    topics_json = json.load(f)

# Map topics in dataframes
df_videos = df_videos.with_columns(
        pl.col("topic").map_elements(
            lambda topic: topics_json.get(topic, ""),
            return_dtype=pl.Utf8
        ).alias(f"BERTopic")
    )
df_commentthreads = df_commentthreads.with_columns(
    pl.col("topic").map_elements(
        lambda topic: topics_json.get(topic, ""),
        return_dtype=pl.Utf8
    ).alias(f"BERTopic")
)
df_commentthreads_replies = df_commentthreads_replies.with_columns(
    pl.col("topic").map_elements(
        lambda topic: topics_json.get(topic, ""),
        return_dtype=pl.Utf8
    ).alias(f"BERTopic")
)

In [33]:
print(f"Videos missing topics: {df_videos.filter(pl.col('BERTopic') == '').height}")
print(f"Comment threads missing topics: {df_commentthreads.filter(pl.col('BERTopic') == '').height}")
print(f"Comment threads replies missing topics: {df_commentthreads_replies.filter(pl.col('BERTopic') == '').height}")

Videos missing topics: 5
Comment threads missing topics: 1836
Comment threads replies missing topics: 795


In [34]:
# Import topic mapping
with open("data/agentopic/mapping.json", "r") as f:
    topic_mapping = json.load(f)

max_levels = max(len(v) for v in topic_mapping.values())

# Map on topicCategories and for each key, create a new column with the mapped value
for level in range(max_levels):
    df_videos = df_videos.with_columns(
        pl.col("BERTopic").map_elements(
            lambda topic: topic_mapping.get(topic, {}).get(f"level_{level}", ""),
            return_dtype=pl.Utf8
        ).alias(f"topic_level_{level}")
    )
    df_commentthreads = df_commentthreads.with_columns(
        pl.col("BERTopic").map_elements(
            lambda topic: topic_mapping.get(topic, {}).get(f"level_{level}", ""),
            return_dtype=pl.Utf8
        ).alias(f"topic_level_{level}")
    )
    df_commentthreads_replies = df_commentthreads_replies.with_columns(
        pl.col("BERTopic").map_elements(
            lambda topic: topic_mapping.get(topic, {}).get(f"level_{level}", ""),
            return_dtype=pl.Utf8
        ).alias(f"topic_level_{level}")
    )

In [41]:
df_videos.filter(pl.col('BERTopic') != '').write_parquet("data/parquet/videos_full_with_topics.parquet")
df_commentthreads.filter(pl.col('BERTopic') != '').write_parquet("data/parquet/commentThreads_full_with_topics.parquet")
df_commentthreads_replies.filter(pl.col('BERTopic') != '').write_parquet("data/parquet/commentThreadsReplies_full_with_topics.parquet")